In [1]:
# Imports
from chessdotcom import *
import pprint
import requests

import urllib.request
import json

import numpy as np
import pandas as pd
import pickle
import json
import time

from datetime import datetime

In [2]:
## Configure Headers, as the Project uses requests package to interact with the API. Headers and paxis can be set
## through the Client Object.
from chessdotcom import Client

Client.request_config["headers"]["User-Agent"] = (
 "Machine learning for chess match outcome Prediction, BSc in Computer Science Dissertation, University of Lincoln"
 "Contact me at conorjackvincent@live.co.uk"
)

Client.rate_limit_handler.tries = 2
Client.rate_limit_handler.tts = 4

In [3]:
## Load the player_profiles_orginal.pkl file, so that we can use this pandas dataframes usernames to collect further statistical data in terms of game history.
with open('Collected_Chess_Data\player_profiles_original.pkl', 'rb') as file:
    player_profiles = pickle.load(file)

In [4]:
player_profiles.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10204 entries, 0 to 354
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          8115 non-null   object
 1   username      10204 non-null  object
 2   title         10204 non-null  object
 3   followers     10204 non-null  int64 
 4   country_code  10204 non-null  object
 5   country       10204 non-null  object
 6   status        10204 non-null  object
 7   is_streamer   10204 non-null  bool  
 8   verified      10204 non-null  bool  
 9   league        8474 non-null   object
dtypes: bool(2), int64(1), object(7)
memory usage: 737.4+ KB


In [5]:
# Create a list containing all the usernames in the username column of player_profiles pandas df.
usernames = player_profiles['username'].tolist()
if 'sanghvipuneet' in usernames:
    print(True)

In [6]:
def obtain_usermame_archives(usernames):
    user_game_archive = {}

    counter = 0 # Counter for displaying progress.
    retry = 0 # Counter for retry attempts.

    
    for username in usernames:
        try:
            # Retrieve player statistics for the current username.
            player_archives_response = get_player_game_archives(username)
            json_structure = player_archives_response.json

            user_game_archive[username] = json_structure
            
            # Print the counter, overwriting the previous counter.
            counter+=1
            print(f'\r{counter} {username}', end='', flush=True)
            
        except ChessDotComError as e:
            # Handle the ChessDotComError and retry
            print(f'\rError for {username}: {e}')
            if retry < 15:
                retry += 1
                print(f'\rRetrying for {username} attempt {retry}...')
                    
            # If max retries reached, move on to the next username
            else:
                print(f'\rMax retries reached for {username}. Moving on.')
                fail_list.append(username)
                retry = 0
                break

    return user_game_archive

In [18]:
user_game_archive = obtain_usermame_archives(usernames)

Error for yardird: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"User \"yardird\" not found."})
Retrying for yardird attempt 1...
Error for alain_otero: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"User \"alain_otero\" not found."})
Retrying for alain_otero attempt 2...
Error for pfizermodernacovid19: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"User \"pfizermodernacovid19\" not found."})
Retrying for pfizermodernacovid19 attempt 3...
Error for chessboy74: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"User \"chessboy74\" not found."})
Retrying for chessboy74 attempt 4...
Error for lizardmorrison: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"User \"lizardmorrison\" not found."})
Retrying for lizardmorrison attempt 5...
Error for puzzle_devil: <class 'chessdotcom.types.ChessDotC

In [7]:
user_game_archive[usernames[7]]

NameError: name 'user_game_archive' is not defined

In [36]:
with open('Collected_Chess_Data/user_game_archives.pkl', 'wb') as file:
    pickle.dump(user_game_archive, file)

In [7]:
with open('Collected_Chess_Data/user_game_archives.pkl', 'rb') as file:
    loaded_archives = pickle.load(file)

In [8]:
loaded_archives[usernames[7]]

{'archives': ['https://api.chess.com/pub/player/anastasiyakarlovych/games/2020/05',
  'https://api.chess.com/pub/player/anastasiyakarlovych/games/2020/06',
  'https://api.chess.com/pub/player/anastasiyakarlovych/games/2020/07',
  'https://api.chess.com/pub/player/anastasiyakarlovych/games/2020/11',
  'https://api.chess.com/pub/player/anastasiyakarlovych/games/2021/06',
  'https://api.chess.com/pub/player/anastasiyakarlovych/games/2021/07',
  'https://api.chess.com/pub/player/anastasiyakarlovych/games/2021/08',
  'https://api.chess.com/pub/player/anastasiyakarlovych/games/2023/12']}

In [9]:
test = loaded_archives[usernames[7]]['archives']

In [10]:
print(test[0])

https://api.chess.com/pub/player/anastasiyakarlovych/games/2020/05


In [11]:
with urllib.request.urlopen(test[0]) as url:
    data = json.load(url)
    print(data)

{'games': [{'url': 'https://www.chess.com/game/live/4902856336', 'pgn': '[Event "Live Chess"]\n[Site "Chess.com"]\n[Date "2020.05.24"]\n[Round "-"]\n[White "anastasiyakarlovych"]\n[Black "Ooo00o"]\n[Result "1-0"]\n[CurrentPosition "1r1b3k/1n1b1N1p/3p3N/p2P4/1pPqP3/1P4Q1/P5PK/5R2 b - -"]\n[Timezone "UTC"]\n[ECO "C30"]\n[ECOUrl "https://www.chess.com/openings/Kings-Gambit"]\n[UTCDate "2020.05.24"]\n[UTCTime "22:00:04"]\n[WhiteElo "2451"]\n[BlackElo "2241"]\n[TimeControl "300"]\n[Termination "anastasiyakarlovych won by checkmate"]\n[StartTime "22:00:04"]\n[EndDate "2020.05.24"]\n[EndTime "22:08:07"]\n[Link "https://www.chess.com/game/live/4902856336"]\n\n1. e4 {[%clk 0:04:42.8]} 1... e5 {[%clk 0:04:50.2]} 2. f4 {[%clk 0:04:39.2]} 2... Ne7 {[%clk 0:04:44.4]} 3. Nf3 {[%clk 0:04:33.4]} 3... Nbc6 {[%clk 0:04:38]} 4. fxe5 {[%clk 0:04:30.5]} 4... Ng6 {[%clk 0:04:34.9]} 5. d4 {[%clk 0:04:28.3]} 5... d5 {[%clk 0:04:32.3]} 6. exd6 {[%clk 0:04:15.8]} 6... cxd6 {[%clk 0:04:32.2]} 7. Bc4 {[%clk 0:04:

In [12]:
# This shows that the loaded url contains a dictionary, with a singular key, games.

with urllib.request.urlopen(test[0]) as url:
    data = json.load(url)
    print(data.keys())

dict_keys(['games'])


In [13]:
# Within the dictionary at key 'games' we have a list, of varying size

with urllib.request.urlopen(test[0]) as url:
    data = json.load(url)
    print(data['games'][3])
    print(type(data['games']))
    print(len(data['games']))

{'url': 'https://www.chess.com/game/live/4902939840', 'pgn': '[Event "Live Chess"]\n[Site "Chess.com"]\n[Date "2020.05.24"]\n[Round "-"]\n[White "anastasiyakarlovych"]\n[Black "michin2009"]\n[Result "1-0"]\n[CurrentPosition "r1bqk2r/ppppbQp1/2n4p/4P3/2B1n3/5N2/PPP2PPP/RNB1K2R b KQkq -"]\n[Timezone "UTC"]\n[ECO "C55"]\n[ECOUrl "https://www.chess.com/openings/Italian-Game-Anti-Fried-Liver-Defense"]\n[UTCDate "2020.05.24"]\n[UTCTime "22:25:40"]\n[WhiteElo "2451"]\n[BlackElo "1085"]\n[TimeControl "300"]\n[Termination "anastasiyakarlovych won by checkmate"]\n[StartTime "22:25:40"]\n[EndDate "2020.05.24"]\n[EndTime "22:26:51"]\n[Link "https://www.chess.com/game/live/4902939840"]\n\n1. e4 {[%clk 0:04:56.3]} 1... e5 {[%clk 0:04:58.6]} 2. Nf3 {[%clk 0:04:53.5]} 2... Nc6 {[%clk 0:04:57.3]} 3. Bc4 {[%clk 0:04:50.2]} 3... h6 {[%clk 0:04:55.5]} 4. d4 {[%clk 0:04:46.7]} 4... Nf6 {[%clk 0:04:53.5]} 5. dxe5 {[%clk 0:04:36]} 5... Nxe4 {[%clk 0:04:47.7]} 6. Qd5 {[%clk 0:04:28.1]} 6... Be7 {[%clk 0:04:28

In [14]:
# Within the list, we have another dictionary which contains a number of keys, Lets determine what are inside each of these and the types that are contained within.
# It is worth noting that the keys inside here can vary, i.e, this may not show all the possible keys. 

with urllib.request.urlopen(test[0]) as url:
    data = json.load(url)
    print(type(data['games'][0]))
    print(data['games'][0].keys())

<class 'dict'>
dict_keys(['url', 'pgn', 'time_control', 'end_time', 'rated', 'tcn', 'uuid', 'initial_setup', 'fen', 'time_class', 'rules', 'white', 'black'])


In [15]:
with urllib.request.urlopen(test[0]) as url:
    data = json.load(url)
    print(data['games'][0].keys())
    for element in data['games'][0].keys():
        print(type(data['games'][0][element]))
        print(data['games'][0][element])

dict_keys(['url', 'pgn', 'time_control', 'end_time', 'rated', 'tcn', 'uuid', 'initial_setup', 'fen', 'time_class', 'rules', 'white', 'black'])
<class 'str'>
https://www.chess.com/game/live/4902856336
<class 'str'>
[Event "Live Chess"]
[Site "Chess.com"]
[Date "2020.05.24"]
[Round "-"]
[White "anastasiyakarlovych"]
[Black "Ooo00o"]
[Result "1-0"]
[CurrentPosition "1r1b3k/1n1b1N1p/3p3N/p2P4/1pPqP3/1P4Q1/P5PK/5R2 b - -"]
[Timezone "UTC"]
[ECO "C30"]
[ECOUrl "https://www.chess.com/openings/Kings-Gambit"]
[UTCDate "2020.05.24"]
[UTCTime "22:00:04"]
[WhiteElo "2451"]
[BlackElo "2241"]
[TimeControl "300"]
[Termination "anastasiyakarlovych won by checkmate"]
[StartTime "22:00:04"]
[EndDate "2020.05.24"]
[EndTime "22:08:07"]
[Link "https://www.chess.com/game/live/4902856336"]

1. e4 {[%clk 0:04:42.8]} 1... e5 {[%clk 0:04:50.2]} 2. f4 {[%clk 0:04:39.2]} 2... Ne7 {[%clk 0:04:44.4]} 3. Nf3 {[%clk 0:04:33.4]} 3... Nbc6 {[%clk 0:04:38]} 4. fxe5 {[%clk 0:04:30.5]} 4... Ng6 {[%clk 0:04:34.9]} 5. d4 {[

In [16]:
import threading
import urllib.request

list_of_game_lists = []
all_urls = []

for k, v in loaded_archives.items():
    for archive in v['archives']:
        all_urls.append(archive)

print(len(all_urls))

427132


In [30]:
import threading
import urllib.request
import json
import time

# Function to fetch data from URL and append game lists to list_of_game_lists
def fetch_and_append(archive, list_of_game_lists, current_index, total_urls):
    while True:
        try:
            with urllib.request.urlopen(archive) as url:
                temp = json.load(url)
                data = temp['games']
                list_of_game_lists.append(data)
            break  # Exit the loop if successful
        except urllib.error.HTTPError as e:
            if e.code == 429:  # Too Many Requests
                print(f"Thread {threading.currentThread().name} encountered HTTP 429 error. Retrying...")
                time.sleep(1)  # Wait for a moment before retrying
            else:
                print(f"Thread {threading.currentThread().name} encountered an error: {e}")
                break  # Exit the loop for other errors

    # Calculate progress percentage
    progress = (current_index + 1) / total_urls * 100
    print(f"Progress: {current_index + 1}/{total_urls} ({progress:.2f}%), Total Threads: {len(threads)}", end='\r', flush=True)

# List to hold threads
threads = []

delay = None

# Iterate over all_urls in pairs and create threads for each pair
for i in range(0, len(all_urls), 10):
    urls_chunk = all_urls[i:i+10]  # Get the next ten URLs
    for j, url in enumerate(urls_chunk):
        thread = threading.Thread(target=fetch_and_append, args=(url, list_of_game_lists, i + j, len(all_urls)), name=f"Thread-{i+j}")
        threads.append(thread)
        thread.start()

    # Wait for all threads to complete before processing the next chunk of URLs
    for thread in threads:
        thread.join()

    # Clear the threads list for the next chunk of URLs
    threads.clear()

KeyboardInterrupt: 

In [23]:
import threading
import urllib.request
import json
import time
import sys

# Initial delay between retry attempts (in seconds)
retry_delay = 0

# Function to fetch data from URL and append game lists to list_of_game_lists
def fetch_and_append(archive, list_of_game_lists):
    global retry_delay  # Access global retry_delay variable
    while True:
        try:
            with urllib.request.urlopen(archive) as url:
                temp = json.load(url)
                data = temp['games']
                list_of_game_lists.append(data)
            break  # Exit the loop if successful
        except urllib.error.HTTPError as e:
            if e.code == 429:  # Too Many Requests
                print(f"Thread {threading.currentThread().name} encountered HTTP 429 error. Retrying...")
                time.sleep(retry_delay)  # Wait for a moment before retrying
                retry_delay *= 1.01  # Increase retry delay by 1%
            else:
                print(f"Thread {threading.currentThread().name} encountered an error: {e}")
                break  # Exit the loop for other errors

# List to hold threads
threads = []

# Iterate over all_urls in pairs and create threads for each pair
for i in range(0, len(all_urls), 10):
    urls_chunk = all_urls[i:i+10]  # Get the next ten URLs

    # Record the start time for each iteration of 10 requests
    start_time = time.time()

    for url in urls_chunk:
        thread = threading.Thread(target=fetch_and_append, args=(url, list_of_game_lists))
        threads.append(thread)
        thread.start()
        
    # Print progress update
    print(f"Progress: {i+1}/{len(all_urls)} ({((i+1)/len(all_urls))*100:.2f}%), Total Threads: {len(threads)}, Retry Delay: {retry_delay:.2f} seconds", end='\r', flush=True)
    
    # Wait for all threads to complete before processing the next chunk of URLs
    for thread in threads:
        thread.join()

    # Calculate the time taken for the current iteration
    end_time = time.time()
    iteration_time = end_time - start_time

    if (i == 0):
        retry_delay = iteration_time

    time.sleep(retry_delay)

    # Clear the threads list for the next chunk of URLs
    threads.clear()

KeyboardInterrupt: 

In [20]:
import threading

# Get the number of active threads
num_threads = threading.active_count()

print(f"Number of active threads: {num_threads}")

Number of active threads: 6


In [16]:
from pyspark import SparkContext, SparkConf

# Initialize SparkContext
conf = SparkConf().setAppName('MapReduce').setMaster('local')
sc = SparkContext(conf = conf)